# Installations & Imports

In [ ]:
!pip install cobra -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.4 MB/s eta 0:00:00


In [ ]:
import cobra

# Anaerobic PlayGround

In [ ]:
bmodel = cobra.io.read_sbml_model("/content/iBsu1147_modify.xml")

ERROR:cobra.io.sbml:'' is not a valid SBML 'SId'.
ERROR:cobra.io.sbml:'R_BTDD-RR' is not a valid SBML 'SId'.
ERROR:cobra.io.sbml:'R_BTDt6-RR' is not a valid SBML 'SId'.


In [ ]:
model = bmodel.copy()

## First i tried to look for the Nitrate Reactions and metabolites

In [ ]:
# Search for nitrate-related reactions
for rxn in model.reactions:
    if "nitrate" in rxn.name.lower() or "nitrate" in rxn.id.lower():
        print(f"ID: {rxn.id}")
        print(f"Name: {rxn.name}")
        print(f"Equation: {rxn.reaction}")
        print(f"Bounds: ({rxn.lower_bound}, {rxn.upper_bound})")
        print("---")

# I mainly care about these two

# ID: EX_no3_e
# Name: Nitrate exchange
# Equation: no3_e -->
# Bounds: (0.0, 1000.0)

# Because this reaction basically allows the bacteria to take in nitrate (currently it is blocked so we need it open)

# ID: NTR3B
# Name: Respiratory nitrate reductase gamma chain (EC 1.7.99.4);Respiratory nitrate reductase beta chain (EC 1.7.99.4)(BSU37270);Respiratory nitrate reductase alpha chain (EC 1.7.99.4)(BSU37280);Respiratory nitrate reductase delta chain (EC 1.7.99.4)(BSU37260)
# Equation: 2.0 h_c + mql7_c + no3_c <=> h2o_c + 2.0 h_e + mqn7_c + no2_c
# Bounds: (-1000.0, 1000.0)

# And this one because this is the respiratory pathway mainly because it acts as an electron acceptor through menaquinol & menaquinone (this one is active so we are ok here)

ID: EX_no3_e
Name: Nitrate exchange
Equation: no3_e --> 
Bounds: (0.0, 1000.0)
---
ID: NTRIR2x
Name: Nitrite reductase [NAD(P)H] small subunit (EC 1.7.1.4);Nitrite reductase [NAD(P)H] large subunit (EC 1.7.1.4)(BSU03300);Assimilatory nitrate reductase large subunit (EC:1.7.99.4)(BSU03310);Nitrite reductase [NAD(P)H] large subunit (EC 1.7.1.4)(BSU03320)
Equation: 5.0 h_c + 3.0 nadh_c + no2_c --> 2.0 h2o_c + 3.0 nad_c + nh4_c
Bounds: (0.0, 1000.0)
---
ID: NTRIR2y
Name: Nitrite reductase [NAD(P)H] small subunit (EC 1.7.1.4);Nitrite reductase [NAD(P)H] large subunit (EC 1.7.1.4)(BSU03300);Assimilatory nitrate reductase large subunit (EC:1.7.99.4)(BSU03310);Nitrite reductase [NAD(P)H] large subunit (EC 1.7.1.4)(BSU03320)
Equation: 5.0 h_c + 3.0 nadph_c + no2_c --> 2.0 h2o_c + 3.0 nadp_c + nh4_c
Bounds: (0.0, 1000.0)
---
ID: NO2t2r
Name: Nitrate/nitrite transporter;Formate/nitrite family of transporters(BSU38060)
Equation: h_e + no2_e <=> h_c + no2_c
Bounds: (-1000.0, 1000.0)
---
ID: NO2t3
N

In [ ]:
for met in model.metabolites:
    if "nitrate" in met.name.lower() or "no3" in met.name.lower():
        print(f"ID: {met.id} | Name: {met.name}")

ID: no3_c | Name: Nitrate
ID: no3_e | Name: Nitrate, extracellular


## Lets activate the nitrate exchnage reaction

In [ ]:
with bmodel as model:
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0    # block oxygen
    model.reactions.get_by_id("EX_no3_e").lower_bound = -1000  # allow nitrate
    sol = model.optimize()
    print(sol.objective_value)

# the bacteria is still dead :(

-2.2455105008799007e-30


## I thought nitrate wasn't being consumed/used

In [ ]:
with bmodel as model:
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0
    model.reactions.get_by_id("EX_no3_e").lower_bound = -1000

    sol = model.optimize()
    print("Growth rate:", sol.objective_value)
    print("Biomass flux:", sol.fluxes["bio00006"])
    print("Nitrate exchange flux:", sol.fluxes["EX_no3_e"])
    print("NTR3B flux:", sol.fluxes["NTR3B"])

# Growth rate: 0.0
# Biomass flux: 0.0
# Nitrate exchange flux: -1.0173913043478346
# NTR3B flux: 1.0173913043478346

# aaaand they are being consumed/used but it is still dead :(

Growth rate: 0.0
Biomass flux: 0.0
Nitrate exchange flux: -1.0173913043478346
NTR3B flux: 1.0173913043478346


## So now let's actually pull the biomass reaction and check the flux coefficinet of each and every molecule of it

In [ ]:
biomass_rxn = model.reactions.get_by_id("bio00006")
print(biomass_rxn.reaction)
print("\nMetabolites:")
for met, coeff in biomass_rxn.metabolites.items():
    print(f"  {coeff:+.4f}  {met.id} ({met.name})")

0.000206831 10fthf_c + 0.000273109 ACP_c + 0.2242 CELLWALL_c + 0.5284 PROTEIN_c + 0.005253 amp_c + 105.003 atp_c + 0.002983 ca2_c + 0.0002924 cdp_c + 0.001153 cmp_c + 0.000127618 coa_c + 0.0655 cpd11462_c + 0.076 cpd15800_c + 0.0006105 ctp_c + 0.026 dna_c + 0.003209 fe3_c + 0.0002215 gdp_c + 0.0005939 gmp_c + 0.0004883 gtp_c + 105.0 h2o_c + 0.6576 k_c + 0.0304 lpteichoic_c + 0.09474 mg2_c + 0.00014977 mql7_c + 0.01822 nad_c + 0.001053 nadp_c + 0.0002367 nadph_c + 0.0008548 ppi_c --> 0.000273109 C03688_c + 104.997 adp_c + 105.0 h_c + 104.987 pi_c

Metabolites:
  -0.0002  10fthf_c (10-Formyltetrahydrofolate)
  -0.0003  ACP_c (Acyl-carrier protein)
  -0.2242  CELLWALL_c (Cell wall of B. subtilis)
  -0.5284  PROTEIN_c (Protein)
  -0.0053  amp_c (AMP)
  -105.0030  atp_c (ATP)
  -0.0030  ca2_c (Calcium)
  -0.0003  cdp_c (CDP)
  -0.0012  cmp_c (CMP)
  -0.0001  coa_c (CoA)
  -0.0655  cpd11462_c (mRNA)
  -0.0760  cpd15800_c (Lipid composition of B. subtilis)
  -0.0006  ctp_c (CTP)
  -0.0260  dn

## tbh i didn't know what to do :D, i gave the list to Claude and it mentioned i should focus on ["nad_c", "nadp_c", "nadph_c", "coa_c", "mql7_c"] since their reactions might just be oxygen dependant

In [ ]:
def find_producing_reactions(model, met_id):
    met = model.metabolites.get_by_id(met_id)
    print(f"\n=== Producers of {met.name} ({met_id}) ===")
    for rxn in met.reactions:
        coeff = rxn.metabolites[met]
        if coeff > 0:  # this reaction produces it
            print(f"  {rxn.id}: {rxn.name}")
            print(f"  Equation: {rxn.reaction}")
            print(f"  Bounds: {rxn.bounds}")
            # Check if O2 is involved
            for r_met in rxn.metabolites:
                if "o2" in r_met.id.lower():
                    print(f"  ⚠️  USES OXYGEN: {r_met.id}")
            print()

for met_id in ["nad_c", "nadp_c", "nadph_c", "coa_c", "mql7_c"]:
    find_producing_reactions(model, met_id)


=== Producers of NAD+ (nad_c) ===
  NODOx: Flavohemoprotein (Hemoglobin-like protein) (Flavohemoglobin) (Nitric oxide dioxygenase) (EC 1.14.12.17)
  Equation: nadh_c + 2.0 no_c + 2.0 o2_c --> h_c + nad_c + 2.0 no3_c
  Bounds: (0.0, 1000.0)
  ⚠️  USES OXYGEN: o2_c

  rxn05433: Enoyl-[acyl-carrier-protein] reductase [NADH] (EC 1.3.1.9);Enoyl-[acyl-carrier-protein] reductase [FMN] (EC 1.3.1.9) homolog(BSU26800)
  Equation: 4methex2eACP_c + h_c + nadh_c --> 4methexACP_c + nad_c
  Bounds: (0.0, 1000.0)

  NMNAT: Nicotinate-nucleotide adenylyltransferase (EC 2.7.7.18)
  Equation: atp_c + nmn_c --> nad_c + ppi_c
  Bounds: (0.0, 1000.0)

  NTRIR2x: Nitrite reductase [NAD(P)H] small subunit (EC 1.7.1.4);Nitrite reductase [NAD(P)H] large subunit (EC 1.7.1.4)(BSU03300);Assimilatory nitrate reductase large subunit (EC:1.7.99.4)(BSU03310);Nitrite reductase [NAD(P)H] large subunit (EC 1.7.1.4)(BSU03320)
  Equation: 5.0 h_c + 3.0 nadh_c + no2_c --> 2.0 h2o_c + 3.0 nad_c + nh4_c
  Bounds: (0.0, 1000.

## So before touching the reactions, i thought maybe i'd check if only one of them is the actual source of death

In [ ]:
# Test which biomass metabolite is the actual bottleneck
problem_mets = ["nad_c", "nadp_c", "nadph_c", "coa_c", "mql7_c"]
biomass = model.reactions.get_by_id("bio00006")

for met_id in problem_mets:
    with bmodel as model:
        # Block O2, open nitrate
        model.reactions.get_by_id("EX_o2_e").lower_bound = 0
        model.reactions.get_by_id("EX_no3_e").lower_bound = -1000

        # Temporarily zero out this metabolite's demand in biomass
        met = model.metabolites.get_by_id(met_id)
        current_coeff = biomass.metabolites.get(met, 0)
        biomass.add_metabolites({met: -current_coeff})  # remove it

        sol = model.optimize()
        print(f"Without {met_id} demand: growth = {sol.objective_value:.6f}")

Without nad_c demand: growth = 0.000000
Without nadp_c demand: growth = 0.000000
Without nadph_c demand: growth = 0.000000
Without coa_c demand: growth = 0.000000
Without mql7_c demand: growth = 0.000000


## What if they are all missing?

In [ ]:
with bmodel as model:
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0
    model.reactions.get_by_id("EX_no3_e").lower_bound = -1000

    biomass = model.reactions.get_by_id("bio00006")
    problem_mets = ["nad_c", "nadp_c", "nadph_c", "coa_c", "mql7_c"]

    for met_id in problem_mets:
        met = model.metabolites.get_by_id(met_id)
        current_coeff = biomass.metabolites.get(met, 0)
        biomass.add_metabolites({met: -current_coeff})

    sol = model.optimize()
    print(f"Without ALL cofactor demands: growth = {sol.objective_value:.6f}")

Without ALL cofactor demands: growth = 0.000000


## Another Dead end, i tried to figure out which reactions that are biomass related are not produced anaerobically by setting them as the objective

In [ ]:
with bmodel as model:
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0
    model.reactions.get_by_id("EX_no3_e").lower_bound = -1000

    biomass = model.reactions.get_by_id("bio00006")

    print("Testing producibility of each biomass component under anaerobic conditions:\n")
    for met, coeff in biomass.metabolites.items():
        if coeff < 0:  # only things the biomass CONSUMES
            # temporarily make this metabolite the objective
            with bmodel as model:
                model.objective = {
                    rxn: rxn.metabolites[met]
                    for rxn in met.reactions
                }
                sol = model.optimize(objective_sense="maximize")
                status = f"{sol.objective_value:.6f}" if sol.status == "optimal" else sol.status
                flag = "❌ STUCK" if sol.status != "optimal" or sol.objective_value < 1e-10 else "✅ OK"
                print(f"{flag}  {met.id:20s} ({met.name}): max producible = {status}")

Testing producibility of each biomass component under anaerobic conditions:

❌ STUCK  10fthf_c             (10-Formyltetrahydrofolate): max producible = 0.000000
❌ STUCK  ACP_c                (Acyl-carrier protein): max producible = 0.000000
❌ STUCK  CELLWALL_c           (Cell wall of B. subtilis): max producible = 0.000000
❌ STUCK  PROTEIN_c            (Protein): max producible = 0.000000
❌ STUCK  amp_c                (AMP): max producible = 0.000000
❌ STUCK  atp_c                (ATP): max producible = 0.000000
❌ STUCK  ca2_c                (Calcium): max producible = 0.000000
❌ STUCK  cdp_c                (CDP): max producible = 0.000000
❌ STUCK  cmp_c                (CMP): max producible = 0.000000
❌ STUCK  cpd11462_c           (mRNA): max producible = 0.000000
❌ STUCK  cpd15800_c           (Lipid composition of B. subtilis): max producible = 0.000000
❌ STUCK  ctp_c                (CTP): max producible = 0.000000
❌ STUCK  dna_c                (DNA): max producible = 0.000000
❌ STUC

## I was totally lost here, tried to check the medium and compare it between the anaerobic and aerobic one

In [ ]:
with bmodel as model:
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0
    model.reactions.get_by_id("EX_no3_e").lower_bound = -1000

    print("Currently open exchange reactions (allowing IMPORT):\n")
    for rxn in model.exchanges:
        if rxn.lower_bound < 0:
            print(f"  {rxn.id:20s} lb={rxn.lower_bound:8.1f}  | {rxn.name}")

    print("\n--- Total open exchanges:",
          sum(1 for r in model.exchanges if r.lower_bound < 0))

Currently open exchange reactions (allowing IMPORT):

  EX_h2o_e             lb= -1000.0  | H2O exchange
  EX_pi_e              lb= -1000.0  | Orthophosphate exchange
  EX_co2_e             lb= -1000.0  | CO2 exchange
  EX_nh4_e             lb= -1000.0  | NH3 exchange
  EX_so4_e             lb= -1000.0  | Sulfate exchange
  EX_ca2_e             lb= -1000.0  | Calcium exchange
  EX_h_e               lb= -1000.0  | H+ exchange
  EX_k_e               lb= -1000.0  | Potassium exchange
  EX_no3_e             lb= -1000.0  | Nitrate exchange
  EX_glc__D_e          lb=    -5.0  | D-Glucose exchange
  EX_mg2_e             lb= -1000.0  | Magnesium exchange
  EX_na1_e             lb= -1000.0  | Sodium exchange
  EX_fe3_e             lb= -1000.0  | Fe3+ exchange

--- Total open exchanges: 13


In [ ]:
# On the original model with no changes
print("Open exchanges in BASE model:\n")
for rxn in model.exchanges:
    if rxn.lower_bound < 0:
        print(f"  {rxn.id:20s} lb={rxn.lower_bound:8.1f}  | {rxn.name}")

Open exchanges in BASE model:

  EX_h2o_e             lb= -1000.0  | H2O exchange
  EX_o2_e              lb= -1000.0  | Oxygen exchange
  EX_pi_e              lb= -1000.0  | Orthophosphate exchange
  EX_co2_e             lb= -1000.0  | CO2 exchange
  EX_nh4_e             lb= -1000.0  | NH3 exchange
  EX_so4_e             lb= -1000.0  | Sulfate exchange
  EX_ca2_e             lb= -1000.0  | Calcium exchange
  EX_h_e               lb= -1000.0  | H+ exchange
  EX_k_e               lb= -1000.0  | Potassium exchange
  EX_glc__D_e          lb=    -5.0  | D-Glucose exchange
  EX_mg2_e             lb= -1000.0  | Magnesium exchange
  EX_na1_e             lb= -1000.0  | Sodium exchange
  EX_fe3_e             lb= -1000.0  | Fe3+ exchange


## I tried to give all the biomass metabolites max coefficients and then drop them one by one in the anaerobic state to figure out which one kills it

In [ ]:
with bmodel as model:
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0
    model.reactions.get_by_id("EX_no3_e").lower_bound = -1000

    biomass = model.reactions.get_by_id("bio00006")

    # First open ALL exchanges again
    for met, coeff in biomass.metabolites.items():
        if coeff < 0:
            ex_id = f"EX_{met.id}"
            if ex_id not in [r.id for r in model.reactions]:
                ex_rxn = cobra.Reaction(ex_id)
                ex_rxn.add_metabolites({met: 1})
                model.add_reactions([ex_rxn])
            model.reactions.get_by_id(ex_id).lower_bound = -1000

    # Now remove one at a time and check growth
    print("Removing each free import one at a time:\n")
    for met, coeff in biomass.metabolites.items():
        if coeff < 0:
            with model:
                # Close just this one exchange
                ex_id = f"EX_{met.id}"
                model.reactions.get_by_id(ex_id).lower_bound = 0

                sol = model.optimize()
                flag = "❌ KILLS GROWTH" if sol.objective_value < 1e-6 else "✅ fine"
                print(f"{flag}  {met.id:20s} ({met.name}): growth = {sol.objective_value:.6f}")

Removing each free import one at a time:

✅ fine  10fthf_c             (10-Formyltetrahydrofolate): growth = 35.659590
✅ fine  ACP_c                (Acyl-carrier protein): growth = 35.659590
✅ fine  CELLWALL_c           (Cell wall of B. subtilis): growth = 35.659590
✅ fine  PROTEIN_c            (Protein): growth = 35.659590
✅ fine  amp_c                (AMP): growth = 35.659590
✅ fine  atp_c                (ATP): growth = 35.659590
✅ fine  ca2_c                (Calcium): growth = 35.659590
✅ fine  cdp_c                (CDP): growth = 35.659590
✅ fine  cmp_c                (CMP): growth = 35.659590
✅ fine  cpd11462_c           (mRNA): growth = 35.659590
✅ fine  cpd15800_c           (Lipid composition of B. subtilis): growth = 35.659590
✅ fine  ctp_c                (CTP): growth = 35.659590
✅ fine  dna_c                (DNA): growth = 35.659590
✅ fine  fe3_c                (Fe3+): growth = 35.659590
✅ fine  gdp_c                (GDP): growth = 35.659590
✅ fine  gmp_c                (GMP)

## Something was off, growth of 35 is unrealistic, since the original aerobic model has growth of 0.39 + it seems like missing one of them didn't kill it at all

## I tried to get a FBA solution for aerobic and one for anaerobic and try to compare them and find the ones that dies out when anaerobic begins + uses oxygen

In [ ]:
sol_aero = model.optimize()
aerobic_fluxes = sol_aero.fluxes

with bmodel as model:
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0
    model.reactions.get_by_id("EX_no3_e").lower_bound = -1000
    sol_anaero = model.optimize()
    anaerobic_fluxes = sol_anaero.fluxes

print("Reactions ACTIVE aerobically, DEAD anaerobically, using ACTUAL O2:\n")
for rxn in model.reactions:
    aero = abs(aerobic_fluxes[rxn.id])
    anaero = abs(anaerobic_fluxes[rxn.id])
    if aero > 1e-6 and anaero < 1e-10:
        # Exact match only
        uses_o2 = any(m.id == "o2_c" for m in rxn.metabolites)
        if uses_o2:
            print(f"  {rxn.id:15s} aero={aero:.4f}")
            print(f"  Equation: {rxn.reaction}")
            print()

Reactions ACTIVE aerobically, DEAD anaerobically, using ACTUAL O2:

  CAT             aero=0.0292
  Equation: 2.0 h2o2_c --> 2.0 h2o_c + o2_c

  ASPO6           aero=0.0077
  Equation: asp__L_c + o2_c --> h2o2_c + h_c + iasp_c

  DHORDfum        aero=0.0506
  Equation: dhor__S_c + o2_c --> h2o2_c + orot_c

  O2t             aero=13.4554
  Equation: o2_e <=> o2_c

  CYOO3           aero=26.8524
  Equation: 2.0 focytc_c + 6.0 h_c + 0.5 o2_c --> 2.0 ficytC_c + h2o_c + 4.0 h_e



I gave the list to Claude, here is what it said

In [ ]:
# CAT — catalase actually produces O2 from H2O2, it's not a consumer. Ignore it.
# O2t — just oxygen transport. Expected to be dead. Ignore it.
# CYOO3 — cytochrome oxidase, the aerobic ETC terminal. Expected to be dead anaerobically. Ignore it.

# That leaves two real culprits:
# ASPO6 — L-aspartate oxidase. This uses O2 to convert aspartate → iminoaspartate, which feeds directly into de novo NAD⁺ biosynthesis (the quinolinate pathway). No O2 → can't make NAD⁺ from scratch.
# DHORDfum — Dihydroorotate dehydrogenase. Uses O2 in pyrimidine biosynthesis — the pathway that makes UMP, which then becomes UTP, CTP, and all the nucleotides the cell needs for RNA and DNA.

## so then we tried to bypass each one alone and check growth and then bypass them both together once more

In [ ]:
with bmodel as model:
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0
    model.reactions.get_by_id("EX_no3_e").lower_bound = -1000

    # Test 1: bypass ASPO6 by opening NAD import
    with bmodel as model:
        met = model.metabolites.get_by_id("nad_c")
        ex = cobra.Reaction("EX_nad_c")
        ex.add_metabolites({met: 1})
        ex.lower_bound = -1000
        model.add_reactions([ex])
        sol = model.optimize()
        print(f"With free NAD import: {sol.objective_value:.6f}")

    # Test 2: bypass DHORDfum by opening orotate import
    with bmodel as model:
        met = model.metabolites.get_by_id("orot_c")
        ex = cobra.Reaction("EX_orot_c")
        ex.add_metabolites({met: 1})
        ex.lower_bound = -1000
        model.add_reactions([ex])
        sol = model.optimize()
        print(f"With free orotate import: {sol.objective_value:.6f}")

    # Test 3: both together
    with bmodel as model:
        for met_id in ["nad_c", "orot_c"]:
            met = model.metabolites.get_by_id(met_id)
            ex = cobra.Reaction(f"EX_{met_id}")
            ex.add_metabolites({met: 1})
            ex.lower_bound = -1000
            model.add_reactions([ex])
        sol = model.optimize()
        print(f"With both free: {sol.objective_value:.6f}")

With free NAD import: 0.000000
With free orotate import: 0.000000
With both free: 0.224092


## IT GROWS :D and 0.22 is around 55%-60% of the aerobic value of 0.39 so this sounds realisitc

## Now the thing was how to have both of them active in a real biological way and not unlimited like we set them in the beggining

## First we look up the DHORD related reactions

In [ ]:
for rxn in model.reactions:
    if "DHORD" in rxn.id or "dihydroorot" in rxn.name.lower():
        print(f"{rxn.id}: {rxn.reaction}")
        print(f"  Bounds: {rxn.bounds}\n")

DHORDfum: dhor__S_c + o2_c --> h2o2_c + orot_c
  Bounds: (0.0, 1000.0)

DHORTSn: dhor__S_c + h2o_c <=> cbasp_c + h_c
  Bounds: (-1000.0, 1000.0)



In [ ]:
# Here is what Claude mentioned

#For pyrimidines — the model is genuinely missing an anaerobic reaction.
# DHORDfum is the only way to convert dihydroorotate → orotate, and it needs O2. But in real B. subtilis,
# there's a second enzyme — menaquinone-dependent DHORD — that does the exact same conversion using menaquinone instead of O2.
#  This reaction exists in the organism but is simply absent from the iBsu1147 model, This is a model gap.

## So then i looked up DHORD reactions on KEGG for B subtilis and found this reaction in B subtilis, it doesn't use menaquinones, it uses NAD and NADH, but they both serve the same function mediating the electron transports

In [ ]:
# REF: https://www.genome.jp/dbget-bin/www_bget?ec:1.3.1.14
# REF: https://www.genome.jp/entry/bsu:BSU15540

dhord_nad = cobra.Reaction("DHORDnad")
dhord_nad.name = "Dihydroorotate dehydrogenase (NAD)"
dhord_nad.bounds = (0, 1000)
dhord_nad.add_metabolites({
    model.metabolites.get_by_id("dhor__S_c"): -1,
    model.metabolites.get_by_id("nad_c"):    -1,
    model.metabolites.get_by_id("orot_c"):    +1,
    model.metabolites.get_by_id("nadh_c"):    +1,
    model.metabolites.get_by_id("h_c"):       +1,
})
model.add_reactions([dhord_nad])

## Then for ASPO6 which is supposed to produce iminoaspartate for the de novo NAD+ synthesis, but it oxygen dependant so i checked KEGG again and found an alternative pathways that depends on NADP as the electron acceptor instead

In [ ]:
# REF: https://www.kegg.jp/entry/K06989
# REF: https://www.kegg.jp/entry/R07407

aspdh = cobra.Reaction("ASPDH")
aspdh.name = "Aspartate dehydrogenase (NADP-dependent)"
aspdh.bounds = (0, 1000)
aspdh.add_metabolites({
    model.metabolites.get_by_id("asp__L_c"): -1,
    model.metabolites.get_by_id("nadp_c"):   -1,
    model.metabolites.get_by_id("iasp_c"):   +1,
    model.metabolites.get_by_id("nadph_c"):  +1,
    model.metabolites.get_by_id("h_c"):      +1,
})
model.add_reactions([aspdh])

## So basically these are the final model changes that we needed for the B subtilis to perform and grow on nitrate respiration

In [ ]:
def bSubtilis_anaerobic_model(model):
    # Block oxygen
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0

    # Open nitrate as alternative electron acceptor
    model.reactions.get_by_id("EX_no3_e").lower_bound = -1000

    # Add NAD-dependent dihydroorotate dehydrogenase
    # Allows pyrimidine synthesis without O2
    # REF: https://www.genome.jp/dbget-bin/www_bget?ec:1.3.1.14
    # REF: https://www.genome.jp/entry/bsu:BSU15540

    dhord_nad = cobra.Reaction("DHORDnad")
    dhord_nad.name = "Dihydroorotate dehydrogenase (NAD)"
    dhord_nad.bounds = (0, 1000)
    dhord_nad.add_metabolites({
        model.metabolites.get_by_id("dhor__S_c"): -1,
        model.metabolites.get_by_id("nad_c"):    -1,
        model.metabolites.get_by_id("orot_c"):    +1,
        model.metabolites.get_by_id("nadh_c"):    +1,
        model.metabolites.get_by_id("h_c"):       +1,
    })
    model.add_reactions([dhord_nad])

    # Add aspartate dehydrogenase for anaerobic NAD+ synthesis
    # REF: https://www.kegg.jp/entry/K06989
    # REF: https://www.kegg.jp/entry/R07407

    aspdh = cobra.Reaction("ASPDH")
    aspdh.name = "Aspartate dehydrogenase (NAD(P)-dependent)"
    aspdh.bounds = (0, 1000)
    aspdh.add_metabolites({
        model.metabolites.get_by_id("asp__L_c"): -1,
        model.metabolites.get_by_id("nadp_c"):   -1,
        model.metabolites.get_by_id("iasp_c"):   +1,
        model.metabolites.get_by_id("nadph_c"):  +1,
        model.metabolites.get_by_id("h_c"):      +1,
    })
    model.add_reactions([aspdh])

    return model

# Yeast

In [ ]:
ymodel = cobra.io.read_sbml_model("/content/yeast-GEM.xml")

In [ ]:
def sCerevisiae_anaerobic_model(model):
    # Based on the original publication's way (https://github.com/SysBioChalmers/yeast-GEM/blob/main/code/otherChanges/anaerobicModel.m)

    # Set NGAM to 0
    model.reactions.get_by_id("r_4046").lower_bound = 0
    model.reactions.get_by_id("r_4046").upper_bound = 0

    # Adjust GAM
    GAM = 30.49
    biomass_rxn = model.reactions.get_by_id("r_4041") # Biomass Psuedoreaction
    gam_mets = {
        "s_0434": -GAM,  # ATP
        "s_0803": -GAM,  # H2O
        "s_0394": GAM,   # ADP
        "s_0794": GAM,   # H+
        "s_1322": GAM,   # phosphate
    }
    for met_id, target_coeff in gam_mets.items():
        met = model.metabolites.get_by_id(met_id)
        current = biomass_rxn.metabolites.get(met, 0)
        if current != target_coeff:
            biomass_rxn.add_metabolites({met: target_coeff - current})

    # Remove O2-dependent cofactors from cofactor pseudoreaction
    # heme a, NAD, NADH, NADP+, NADPH, coenzyme A, not used anaerobically
    cofactor_rxn = model.reactions.get_by_id("r_4598")
    mets_to_remove = ['s_3714', 's_1198', 's_1203', 's_1207', 's_1212', 's_0529']
    for met_id in mets_to_remove:
        met = model.metabolites.get_by_id(met_id)
        current = cofactor_rxn.metabolites.get(met, 0)
        if current != 0:
            cofactor_rxn.add_metabolites({met: -current})  # zero it out

    # Anaerobic media
    model.reactions.get_by_id("r_1992").lower_bound = 0       # block O2
    model.reactions.get_by_id("r_1757").lower_bound = -1000   # ergosterol
    model.reactions.get_by_id("r_1915").lower_bound = -1000   # lanosterol
    model.reactions.get_by_id("r_1994").lower_bound = -1000   # palmitoleate
    model.reactions.get_by_id("r_2106").lower_bound = -1000   # zymosterol
    model.reactions.get_by_id("r_2134").lower_bound = -1000   # 14-demethyllanosterol
    model.reactions.get_by_id("r_2189").lower_bound = -1000   # oleate
    model.reactions.get_by_id("r_2137").lower_bound = -1000   # ergosta-5,7,22,24(28)-tetraen-3beta-ol

    # Block pathways
    model.reactions.get_by_id("r_0713").lower_bound = 0   # oxaloacetate-malate shuttle (Mithocondria)
    model.reactions.get_by_id("r_0714").lower_bound = 0   # oxaloacetate-malate shuttle (Cytoplasm)
    model.reactions.get_by_id("r_0487").upper_bound = 0   # glycerol dehydrogenase
    model.reactions.get_by_id("r_0472").upper_bound = 0   # alternative glutamate pathway

    return model

In [ ]:
a_ymodel = sCerevisiae_anaerobic_model(ymodel)

In [ ]:
print(f'Aerobic Growth: {ymodel.slim_optimize()}')
print(f'Anaerobic Growth: {a_ymodel.slim_optimize()}')

0.023786417604888317

# Bacillus

In [ ]:
bmodel = cobra.io.read_sbml_model("/content/iBsu1147_modify.xml")

ERROR:cobra.io.sbml:'' is not a valid SBML 'SId'.
ERROR:cobra.io.sbml:'R_BTDD-RR' is not a valid SBML 'SId'.
ERROR:cobra.io.sbml:'R_BTDt6-RR' is not a valid SBML 'SId'.


In [ ]:
def bSubtilis_anaerobic_model(model):
    # Block oxygen
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0

    # Open nitrate as alternative electron acceptor
    model.reactions.get_by_id("EX_no3_e").lower_bound = -1000

    # Add NAD-dependent dihydroorotate dehydrogenase
    # Allows pyrimidine synthesis without O2
    # REF: https://www.genome.jp/dbget-bin/www_bget?ec:1.3.1.14
    # REF: https://www.genome.jp/entry/bsu:BSU15540

    dhord_nad = cobra.Reaction("DHORDnad")
    dhord_nad.name = "Dihydroorotate dehydrogenase (NAD)"
    dhord_nad.bounds = (0, 1000)
    dhord_nad.add_metabolites({
        model.metabolites.get_by_id("dhor__S_c"): -1,
        model.metabolites.get_by_id("nad_c"):    -1,
        model.metabolites.get_by_id("orot_c"):    +1,
        model.metabolites.get_by_id("nadh_c"):    +1,
        model.metabolites.get_by_id("h_c"):       +1,
    })
    model.add_reactions([dhord_nad])

    # Add aspartate dehydrogenase for anaerobic NAD+ synthesis
    # REF: https://www.kegg.jp/entry/K06989
    # REF: https://www.kegg.jp/entry/R07407

    aspdh = cobra.Reaction("ASPDH")
    aspdh.name = "Aspartate dehydrogenase (NADP-dependent)"
    aspdh.bounds = (0, 1000)
    aspdh.add_metabolites({
        model.metabolites.get_by_id("asp__L_c"): -1,
        model.metabolites.get_by_id("nadp_c"):   -1,
        model.metabolites.get_by_id("iasp_c"):   +1,
        model.metabolites.get_by_id("nadph_c"):  +1,
        model.metabolites.get_by_id("h_c"):      +1,
    })
    model.add_reactions([aspdh])

    return model

In [ ]:
a_bmodel = bSubtilis_anaerobic_model(bmodel)

In [ ]:
print(f'Aerobic Growth: {bmodel.slim_optimize()}')
print(f'Anaerobic Growth: {a_bmodel.slim_optimize()}')

Aerobic Growth: 0.39502675579421165
Anaerobic Growth: 0.22171481928255005


# Key Takeaways

## Yeast9 Anaerobize Function

In [ ]:
def sCerevisiae_anaerobic_model(model):
    # Based on the original publication's way (https://github.com/SysBioChalmers/yeast-GEM/blob/main/code/otherChanges/anaerobicModel.m)

    # Set NGAM to 0
    model.reactions.get_by_id("r_4046").lower_bound = 0
    model.reactions.get_by_id("r_4046").upper_bound = 0

    # Adjust GAM
    GAM = 30.49
    biomass_rxn = model.reactions.get_by_id("r_4041") # Biomass Psuedoreaction
    gam_mets = {
        "s_0434": -GAM,  # ATP
        "s_0803": -GAM,  # H2O
        "s_0394": GAM,   # ADP
        "s_0794": GAM,   # H+
        "s_1322": GAM,   # phosphate
    }
    for met_id, target_coeff in gam_mets.items():
        met = model.metabolites.get_by_id(met_id)
        current = biomass_rxn.metabolites.get(met, 0)
        if current != target_coeff:
            biomass_rxn.add_metabolites({met: target_coeff - current})

    # Remove O2-dependent cofactors from cofactor pseudoreaction
    # heme a, NAD, NADH, NADP+, NADPH, coenzyme A, not used anaerobically
    cofactor_rxn = model.reactions.get_by_id("r_4598")
    mets_to_remove = ['s_3714', 's_1198', 's_1203', 's_1207', 's_1212', 's_0529']
    for met_id in mets_to_remove:
        met = model.metabolites.get_by_id(met_id)
        current = cofactor_rxn.metabolites.get(met, 0)
        if current != 0:
            cofactor_rxn.add_metabolites({met: -current})  # zero it out

    # Anaerobic media
    model.reactions.get_by_id("r_1992").lower_bound = 0       # block O2
    model.reactions.get_by_id("r_1757").lower_bound = -1000   # ergosterol
    model.reactions.get_by_id("r_1915").lower_bound = -1000   # lanosterol
    model.reactions.get_by_id("r_1994").lower_bound = -1000   # palmitoleate
    model.reactions.get_by_id("r_2106").lower_bound = -1000   # zymosterol
    model.reactions.get_by_id("r_2134").lower_bound = -1000   # 14-demethyllanosterol
    model.reactions.get_by_id("r_2189").lower_bound = -1000   # oleate
    model.reactions.get_by_id("r_2137").lower_bound = -1000   # ergosta-5,7,22,24(28)-tetraen-3beta-ol

    # Block pathways
    model.reactions.get_by_id("r_0713").lower_bound = 0   # oxaloacetate-malate shuttle (Mithocondria)
    model.reactions.get_by_id("r_0714").lower_bound = 0   # oxaloacetate-malate shuttle (Cytoplasm)
    model.reactions.get_by_id("r_0487").upper_bound = 0   # glycerol dehydrogenase
    model.reactions.get_by_id("r_0472").upper_bound = 0   # alternative glutamate pathway

    return model

## iBsu1147 Anaerobize Function

In [ ]:
def bSubtilis_anaerobic_model(model):
    # Block oxygen
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0

    # Open nitrate as alternative electron acceptor
    model.reactions.get_by_id("EX_no3_e").lower_bound = -1000

    # Add NAD-dependent dihydroorotate dehydrogenase
    # Allows pyrimidine synthesis without O2
    # REF: https://www.genome.jp/dbget-bin/www_bget?ec:1.3.1.14
    # REF: https://www.genome.jp/entry/bsu:BSU15540

    dhord_nad = cobra.Reaction("DHORDnad")
    dhord_nad.name = "Dihydroorotate dehydrogenase (NAD)"
    dhord_nad.bounds = (0, 1000)
    dhord_nad.add_metabolites({
        model.metabolites.get_by_id("dhor__S_c"): -1,
        model.metabolites.get_by_id("nad_c"):    -1,
        model.metabolites.get_by_id("orot_c"):    +1,
        model.metabolites.get_by_id("nadh_c"):    +1,
        model.metabolites.get_by_id("h_c"):       +1,
    })
    model.add_reactions([dhord_nad])

    # Add aspartate dehydrogenase for anaerobic NAD+ synthesis
    # REF: https://www.kegg.jp/entry/K06989
    # REF: https://www.kegg.jp/entry/R07407

    aspdh = cobra.Reaction("ASPDH")
    aspdh.name = "Aspartate dehydrogenase (NADP-dependent)"
    aspdh.bounds = (0, 1000)
    aspdh.add_metabolites({
        model.metabolites.get_by_id("asp__L_c"): -1,
        model.metabolites.get_by_id("nadp_c"):   -1,
        model.metabolites.get_by_id("iasp_c"):   +1,
        model.metabolites.get_by_id("nadph_c"):  +1,
        model.metabolites.get_by_id("h_c"):      +1,
    })
    model.add_reactions([aspdh])

    return model